# LeetCode 299 — Bulls and Cows bằng Python

Link đề: https://leetcode.com/problems/bulls-and-cows/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Counting

Skeleton LeetCode:

```python
class Solution:
    def getHint(self, secret: str, guess: str) -> str:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ file đề có sẵn (bài này không phải Premium). Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với brute force độc lập (thử mọi cách sắp xếp lại các chữ số không-bull của lời đoán để tối đa hóa số chữ số khớp).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: so sánh chuỗi bí mật `secret` với lời đoán `guess` (cùng độ dài, chỉ chữ số) và trả gợi ý dạng `"xAyB"` với `x` là số **bulls** (đúng chữ số **và** đúng vị trí) và `y` là số **cows** (đúng chữ số nhưng **sai vị trí**, mỗi chữ số của `secret` chỉ được dùng **một lần**).
2. Hiểu cách xử lý **chữ số trùng lặp**: một chữ số của `secret` không thể "khớp" cho hai chữ số của `guess`.
3. Đi từ cách **loại bulls rồi ghép từng chữ số còn lại bằng danh sách** `O(n²)` → **`Counter`: `sum(min(cnt_s, cnt_g)) - bulls`** → **mảng đếm 10 phần tử một lượt (cân bằng ± )** `O(n)` → **sắp xếp các chữ số còn lại rồi hai con trỏ**.
4. Nắm công cụ `Counter`, `zip`, mảng đếm theo chữ số và công thức "**giao của hai đa tập**".
5. Tránh các lỗi kinh điển: tính cow bằng **`in`/tập hợp** (bỏ qua số lần xuất hiện), **đếm lại bulls** trong cows, nhầm **định dạng** `xAyB`, và dùng sai `Counter` (cộng thay vì lấy `min`).

## 2. Đề bài & ví dụ

Bạn chơi trò **Bulls and Cows** với bạn: bạn viết ra một **số bí mật** và bạn của bạn đoán. Mỗi lần đoán, bạn đưa gợi ý:

- **Bulls (bò đực)**: số chữ số trong lời đoán đúng **vị trí**.
- **Cows (bò cái)**: số chữ số trong lời đoán **có trong số bí mật nhưng ở sai vị trí**. Chính xác hơn: các chữ số **không phải bull** của lời đoán mà có thể **sắp xếp lại** để trở thành bull.

Cho `secret` và `guess`, trả về gợi ý theo dạng `"xAyB"` (`x` bulls, `y` cows). Chú ý cả hai có thể chứa **chữ số trùng lặp**.

**Ví dụ 1**

- Input: `secret = "1807", guess = "7810"`
- Output: `"1A3B"`
- Giải thích: bull là `8` ở vị trí `1`. Các cow là `1`, `0`, `7` (có trong bí mật nhưng sai vị trí).

**Ví dụ 2**

- Input: `secret = "1123", guess = "0111"`
- Output: `"1A1B"`
- Giải thích: bull là `1` ở vị trí `1`. Trong số các chữ số còn lại của lời đoán (`0`, `1`, `1`), chỉ **một** chữ số `1` được tính là cow, vì bí mật còn đúng một chữ số `1` chưa dùng.

**Ràng buộc:** `1 <= secret.length, guess.length <= 1000`; `secret.length == guess.length`; `secret` và `guess` chỉ gồm chữ số.

In [1]:
from typing import List
from collections import Counter
from itertools import permutations, product
import random
import time

examples = [
    ("1807", "7810", "1A3B"),
    ("1123", "0111", "1A1B"),
]
for secret, guess, expected in examples:
    print(f"secret={secret!r}, guess={guess!r} -> kỳ vọng {expected!r}")

secret='1807', guess='7810' -> kỳ vọng '1A3B'
secret='1123', guess='0111' -> kỳ vọng '1A1B'


## 3. Kiến thức nền

### 3.1. Duyệt hai chuỗi song song: `zip`

`zip(a, b)` ghép phần tử cùng vị trí của hai chuỗi thành các cặp. Dùng để so sánh `secret[i]` với `guess[i]` mà không cần chỉ số.
#### Code demo

In [2]:
for i, (s, g) in enumerate(zip("1807", "7810")):
    print(i, s, g, "bull" if s == g else "")
print(sum(1 for s, g in zip("1807", "7810") if s == g))        # 1 bull

0 1 7 
1 8 8 bull
2 0 1 
3 7 0 
1


### 3.2. `Counter`: đa tập các ký tự

`Counter("1123")` trả `{'1': 2, '2': 1, '3': 1}`: số lần mỗi ký tự xuất hiện (**đa tập — multiset**). `Counter` hỗ trợ toán tử `&` (**giao đa tập**, lấy `min` số lần từng phần tử) và `|` (hợp, lấy `max`). `sum((a & b).values())` là **kích thước giao** của hai đa tập.
#### Code demo

In [3]:
a, b = Counter("1123"), Counter("0111")
print(a, b)
print(a & b, sum((a & b).values()))          # Counter({'1': 2}) 2: có thể ghép tối đa 2 chữ số '1'
print(a | b)

Counter({'1': 2, '2': 1, '3': 1}) Counter({'1': 3, '0': 1})
Counter({'1': 2}) 2
Counter({'1': 3, '2': 1, '3': 1, '0': 1})


### 3.3. Tại sao cần "số lần": ghép đúng một–một

Mỗi chữ số của `guess` chỉ ghép với **một** chữ số của `secret` và ngược lại. Với chữ số `d`, nếu `secret` có `p` chữ số `d` và `guess` có `q` chữ số `d` thì tối đa **`min(p, q)`** chữ số `d` được ghép (bull hoặc cow). Tổng `Σ_d min(p_d, q_d)` là tổng số cặp ghép; trừ đi bulls ta được cows.
#### Code demo

In [4]:
secret, guess = "1123", "0111"
total_matches = sum(min(Counter(secret)[d], Counter(guess)[d]) for d in "0123456789")
bulls = sum(1 for s, g in zip(secret, guess) if s == g)
print("tổng cặp ghép:", total_matches, "bulls:", bulls, "cows:", total_matches - bulls)     # 2, 1, 1

tổng cặp ghép: 2 bulls: 1 cows: 1


### 3.4. Định dạng chuỗi với f-string

`f"{bulls}A{cows}B"` ghép số và chữ thành chuỗi kết quả, ví dụ `f"{1}A{3}B"` là `"1A3B"`.
#### Code demo

In [5]:
bulls, cows = 1, 3
print(f"{bulls}A{cows}B")                     # 1A3B
print(str(bulls) + "A" + str(cows) + "B")     # cách khác

1A3B
1A3B


### 3.5. Mảng đếm theo chữ số

Chữ số chỉ có `10` giá trị `0..9` nên có thể dùng danh sách `cnt = [0] * 10` thay cho `Counter`; chỉ số là `int(ch)` (hoặc `ord(ch) - ord("0")`). Nhanh và gọn.
#### Code demo

In [6]:
cnt = [0] * 10
for ch in "1123":
    cnt[int(ch)] += 1
print(cnt)                                    # [0, 2, 1, 1, 0, 0, 0, 0, 0, 0]

[0, 2, 1, 1, 0, 0, 0, 0, 0, 0]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 2: `secret = "1123"`, `guess = "0111"`.

1. **Bulls:** so từng vị trí: `1 vs 0`, `1 vs 1` (bull!), `2 vs 1`, `3 vs 1`. Có `1` bull.
2. **Loại bull:** bỏ vị trí `1` khỏi cả hai. Còn `secret' = "1", "2", "3"` (các chữ số `1, 2, 3`) và `guess' = "0", "1", "1"`.
3. **Ghép cow:** `guess'` có `0` (không có trong `secret'`), `1` (khớp với `1` của `secret'`), `1` thứ hai (không còn chữ số `1` nào trong `secret'` để ghép). Vậy `1` cow.

Kết quả `1A1B`. **Cách nghĩ:** bulls là phép so sánh theo vị trí; cows là **giao đa tập** của các chữ số còn lại sau khi loại bulls. Hoặc gộp: cows = (giao đa tập của **toàn bộ** chữ số) − bulls.

## 5. Cách 1 — Brute Force / baseline: loại bulls rồi ghép từng chữ số bằng danh sách

**Ý tưởng.** Đếm bulls và đưa các cặp không phải bull vào hai danh sách `rest_s`, `rest_g`. Với mỗi chữ số trong `rest_g`, nếu nó có trong `rest_s` thì **xóa một bản** khỏi `rest_s` và cộng một cow.
#### Code demo

In [7]:
def get_hint_list(secret, guess):
    bulls = 0
    rest_s, rest_g = [], []
    for s, g in zip(secret, guess):
        if s == g:
            bulls += 1
        else:
            rest_s.append(s)
            rest_g.append(g)
    cows = 0
    for g in rest_g:
        if g in rest_s:
            rest_s.remove(g)                         # dùng chữ số này đúng một lần
            cows += 1
    return f"{bulls}A{cows}B"


for secret, guess, expected in examples:
    print(secret, guess, "->", get_hint_list(secret, guess), "(kỳ vọng", expected, ")")

1807 7810 -> 1A3B (kỳ vọng 1A3B )
1123 0111 -> 1A1B (kỳ vọng 1A1B )


**Độ phức tạp (Cách 1).** Gọi `n = len(secret)`. `g in rest_s` và `remove` đều `O(n)`; vòng lặp `n` lần ⇒ **Time `O(n²)`**; với `n = 1000` là `~10^6` — vẫn chạy được nhưng chậm so với `O(n)`; **Space `O(n)`**.

**Nút thắt.** Tìm kiếm tuyến tính trong danh sách. Vì chữ số chỉ có `10` giá trị, dùng **mảng đếm** thay cho danh sách.

## 6. Ý tưởng thuật toán / bước đột phá

**Công thức giao đa tập.** `bulls = số vị trí s[i] == g[i]`. Tổng số cặp ghép tối đa giữa các chữ số là `T = Σ_d min(cnt_s[d], cnt_g[d])`. Trong `T` cặp này, `bulls` cặp là bull (vì bull cũng là cặp ghép hợp lệ cho chữ số của nó), phần còn lại là cow: **`cows = T − bulls`**.

**Một lượt với mảng cân bằng:** duy trì `cnt[d] = (số d của secret đã thấy chưa ghép) − (số d của guess đã thấy chưa ghép)`. Với mỗi cặp không phải bull `(s, g)`: nếu `cnt[s] < 0` (trước đó có `g` chờ chữ số `s`) thì được một cow; nếu `cnt[g] > 0` (trước đó có `s` chờ chữ số `g`) thì được một cow; rồi `cnt[s] += 1`, `cnt[g] -= 1`. Mỗi chữ số xử lý `O(1)` ⇒ `O(n)`.

## 7. Các cách tiếp theo

### Cách 2 — `Counter` và giao đa tập (cách nộp chuẩn, ngắn gọn)

**Giải thích.** `bulls = sum(a == b for a, b in zip(secret, guess))`; `common = sum((Counter(secret) & Counter(guess)).values())`; `cows = common - bulls`.
#### Code demo

In [8]:
def get_hint_counter(secret, guess):
    bulls = sum(a == b for a, b in zip(secret, guess))
    common = sum((Counter(secret) & Counter(guess)).values())      # kích thước giao đa tập
    return f"{bulls}A{common - bulls}B"


for secret, guess, expected in examples:
    print(secret, guess, "->", get_hint_counter(secret, guess), "(kỳ vọng", expected, ")")

1807 7810 -> 1A3B (kỳ vọng 1A3B )
1123 0111 -> 1A1B (kỳ vọng 1A1B )


In [9]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def getHint(self, secret: str, guess: str) -> str:
        cnt1, cnt2 = Counter(), Counter()
        x = 0
        for a, b in zip(secret, guess):
            if a == b:
                x += 1
            else:
                cnt1[a] += 1
                cnt2[b] += 1
        y = sum(min(cnt1[c], cnt2[c]) for c in cnt1)
        return f"{x}A{y}B"


print(Solution().getHint("1807", "7810"))   # 1A3B

1A3B


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(1)`** (tối đa `10` khóa; `O(k)` với `k = 10`). Bản nộp ở trên đếm riêng các chữ số **không phải bull** (`cnt1`, `cnt2`) rồi lấy `min`, tương đương `common - bulls`.

### Cách 3 — Một lượt, mảng đếm cân bằng

**Giải thích.** Như mục 6: `cnt` có `10` ô; mỗi cặp không phải bull cập nhật theo luật "nhìn xem có chữ số chờ sẵn không".
#### Code demo

In [10]:
def get_hint_one_pass(secret, guess):
    bulls = cows = 0
    cnt = [0] * 10                                    # cnt[d] > 0: d của secret đang chờ; < 0: d của guess đang chờ
    for s, g in zip(secret, guess):
        if s == g:
            bulls += 1
        else:
            si, gi = int(s), int(g)
            if cnt[si] < 0:                           # trước đó có một g == s đang chờ: ghép thành cow
                cows += 1
            if cnt[gi] > 0:                           # trước đó có một s == g đang chờ: ghép thành cow
                cows += 1
            cnt[si] += 1
            cnt[gi] -= 1
    return f"{bulls}A{cows}B"


for secret, guess, expected in examples:
    print(secret, guess, "->", get_hint_one_pass(secret, guess), "(kỳ vọng", expected, ")")

1807 7810 -> 1A3B (kỳ vọng 1A3B )
1123 0111 -> 1A1B (kỳ vọng 1A1B )


**Độ phức tạp Cách 3:** **Time `O(n)`** (một lượt, mỗi bước `O(1)`), **Space `O(1)`** (mảng `10` ô).

### Cách 4 — Hai mảng đếm rồi cộng `min`

**Giải thích.** Đếm bulls; đếm riêng số lần xuất hiện của từng chữ số trong `secret` và `guess` (mảng `10` ô); `cows = Σ_d min(cs[d], cg[d]) − bulls`.
#### Code demo

In [11]:
def get_hint_two_arrays(secret, guess):
    bulls = 0
    cs, cg = [0] * 10, [0] * 10
    for s, g in zip(secret, guess):
        if s == g:
            bulls += 1
        cs[int(s)] += 1
        cg[int(g)] += 1
    return f"{bulls}A{sum(min(a, b) for a, b in zip(cs, cg)) - bulls}B"


for secret, guess, expected in examples:
    print(secret, guess, "->", get_hint_two_arrays(secret, guess), "(kỳ vọng", expected, ")")

1807 7810 -> 1A3B (kỳ vọng 1A3B )
1123 0111 -> 1A1B (kỳ vọng 1A1B )


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 5 — Sắp xếp các chữ số còn lại và hai con trỏ

**Giải thích.** Sau khi loại bulls, sắp xếp `rest_s` và `rest_g`; dùng hai con trỏ như phép **giao hai dãy đã sắp** (đẩy con trỏ của số nhỏ hơn; khi bằng nhau thì đếm một cow và đẩy cả hai).
#### Code demo

In [12]:
def get_hint_sorted(secret, guess):
    bulls = 0
    rest_s, rest_g = [], []
    for s, g in zip(secret, guess):
        if s == g:
            bulls += 1
        else:
            rest_s.append(s)
            rest_g.append(g)
    rest_s.sort()
    rest_g.sort()
    i = j = cows = 0
    while i < len(rest_s) and j < len(rest_g):
        if rest_s[i] == rest_g[j]:
            cows += 1
            i += 1
            j += 1
        elif rest_s[i] < rest_g[j]:
            i += 1
        else:
            j += 1
    return f"{bulls}A{cows}B"


for secret, guess, expected in examples:
    print(secret, guess, "->", get_hint_sorted(secret, guess), "(kỳ vọng", expected, ")")

1807 7810 -> 1A3B (kỳ vọng 1A3B )
1123 0111 -> 1A1B (kỳ vọng 1A1B )


**Độ phức tạp Cách 5:** **Time `O(n log n)`** (sắp xếp), **Space `O(n)`**. Hữu ích khi muốn tránh bảng băm hoặc khi bộ ký tự lớn.

> Cách 2 (ngắn) và Cách 3 (một lượt `O(1)` bộ nhớ) là hai bản nộp tốt. Cách 1 làm baseline.

## 8. TRACE từng bước

Ta trace Cách 3 (một lượt) trên ví dụ 2 `"1123"` vs `"0111"` và ví dụ 1, in `cnt` và số cow cộng thêm tại mỗi vị trí.
#### Code demo

In [13]:
def trace_one_pass(secret, guess):
    print(f"secret={secret!r}, guess={guess!r}")
    bulls = cows = 0
    cnt = [0] * 10
    for i, (s, g) in enumerate(zip(secret, guess)):
        if s == g:
            bulls += 1
            print(f"  i={i}: {s} == {g}: BULL (bulls={bulls})")
            continue
        si, gi = int(s), int(g)
        gain = 0
        notes = []
        if cnt[si] < 0:
            gain += 1
            notes.append(f"cnt[{si}]={cnt[si]} < 0: có '{s}' của guess đang chờ -> cow")
        if cnt[gi] > 0:
            gain += 1
            notes.append(f"cnt[{gi}]={cnt[gi]} > 0: có '{g}' của secret đang chờ -> cow")
        cows += gain
        cnt[si] += 1
        cnt[gi] -= 1
        shown = {d: v for d, v in enumerate(cnt) if v}
        print(f"  i={i}: s={s}, g={g}: +{gain} cow; {'; '.join(notes) if notes else 'không ghép được ngay'}; cnt={shown}")
    print("kết quả:", f"{bulls}A{cows}B")
    return f"{bulls}A{cows}B"


trace_one_pass("1123", "0111")
print()
trace_one_pass("1807", "7810")

secret='1123', guess='0111'
  i=0: s=1, g=0: +0 cow; không ghép được ngay; cnt={0: -1, 1: 1}
  i=1: 1 == 1: BULL (bulls=1)
  i=2: s=2, g=1: +1 cow; cnt[1]=1 > 0: có '1' của secret đang chờ -> cow; cnt={0: -1, 2: 1}
  i=3: s=3, g=1: +0 cow; không ghép được ngay; cnt={0: -1, 1: -1, 2: 1, 3: 1}
kết quả: 1A1B

secret='1807', guess='7810'
  i=0: s=1, g=7: +0 cow; không ghép được ngay; cnt={1: 1, 7: -1}
  i=1: 8 == 8: BULL (bulls=1)
  i=2: s=0, g=1: +1 cow; cnt[1]=1 > 0: có '1' của secret đang chờ -> cow; cnt={0: 1, 7: -1}
  i=3: s=7, g=0: +2 cow; cnt[7]=-1 < 0: có '7' của guess đang chờ -> cow; cnt[0]=1 > 0: có '0' của secret đang chờ -> cow; cnt={}
kết quả: 1A3B


'1A3B'

Bảng tóm tắt (ví dụ 2):

| `i` | `(s, g)` | loại | `cnt` sau | cows |
|---|---|---|---|---|
| 0 | `(1, 0)` | không bull | `{1: +1, 0: -1}` | 0 |
| 1 | `(1, 1)` | **bull** | không đổi | 0 |
| 2 | `(2, 1)` | không bull | `cnt[g=1] = +1 > 0` (có chữ số `1` của secret đang chờ) ⇒ **+1 cow**; sau đó `cnt = {0: -1, 2: +1}` (`cnt[1]` về `0`) | 1 |
| 3 | `(3, 1)` | không bull | `cnt[3] = 0`, `cnt[1] = 0`: không ghép; `cnt = {0: -1, 1: -1, 2: +1, 3: +1}` | 1 |

Kết quả `1A1B`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (đếm các chữ số không-bull). Đúng `class Solution`, đúng `getHint(self, secret: str, guess: str) -> str`.

In [14]:
class Solution:
    def getHint(self, secret: str, guess: str) -> str:
        cnt1, cnt2 = Counter(), Counter()                 # đếm chữ số KHÔNG phải bull ở mỗi chuỗi
        x = 0                                             # bulls
        for a, b in zip(secret, guess):
            if a == b:
                x += 1
            else:
                cnt1[a] += 1
                cnt2[b] += 1
        y = sum(min(cnt1[c], cnt2[c]) for c in cnt1)      # cows: ghép tối đa mỗi chữ số, mỗi chữ số một lần
        return f"{x}A{y}B"


sol = Solution()
for secret, guess, expected in examples:
    assert sol.getHint(secret, guess) == expected
print("OK: 2 ví dụ của đề")

OK: 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Bulls:** một vị trí là bull ⟺ `secret[i] == guess[i]`; đếm trực tiếp.
- **Cows = giao đa tập của phần không-bull:** theo đề, cow là số chữ số không-bull của `guess` có thể sắp xếp lại thành bull, tức là **số cặp ghép tối đa** giữa các chữ số không-bull của `guess` và `secret` thỏa cùng chữ số. Với từng chữ số `d`, có `p` bản `d` không-bull trong `secret` và `q` bản trong `guess`; ghép một–một được tối đa `min(p, q)` cặp; các chữ số khác nhau độc lập ⇒ tổng `Σ_d min(p_d, q_d)`.
- **Vì sao bulls không bị đếm lại:** bản nộp chỉ đếm các chữ số **không-bull** vào `cnt1`, `cnt2`; công thức thay thế `Σ min(toàn bộ) − bulls` cũng đúng vì với mỗi chữ số, cặp bull dùng một bản `d` ở mỗi bên và `min(p_d + b_d, q_d + b_d) = min(p_d, q_d) + b_d` (với `b_d` số bull của chữ số `d`).
- **Cách 3 (một lượt):** `cnt[d]` là số dư của `d` bên `secret` trừ bên `guess` trong các cặp không-bull đã thấy; khi một chữ số mới tới khớp với phần dư bên đối diện thì tạo đúng một cặp ghép (cow); đây là thuật toán ghép tham lam trực tuyến cho giao đa tập.
- **Cách 5:** hai con trỏ trên hai dãy đã sắp đếm đúng kích thước giao đa tập.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(secret)` (tối đa `1000`).

| Cách | Time | Space |
|---|---|---|
| 1. Danh sách + `remove` | `O(n²)` | `O(n)` |
| 2. `Counter` | `O(n)` | `O(1)` (≤ 10 khóa) |
| 3. Một lượt, mảng cân bằng | `O(n)` | `O(1)` |
| 4. Hai mảng đếm | `O(n)` | `O(1)` |
| 5. Sắp xếp + hai con trỏ | `O(n log n)` | `O(n)` |

Với `n = 1000` mọi cách đều tức thời; sự khác biệt thể hiện khi `n` rất lớn (`10^6`). Không có khác biệt average/worst (luôn duyệt hết hai chuỗi).

## 12. Bộ test

`oracle` là brute force độc lập theo **định nghĩa của đề**: bulls đếm theo vị trí; cows = số chữ số không-bull của `guess` mà ta có thể làm thành bull bằng cách **hoán vị** các chữ số không-bull đó: thử **mọi hoán vị** của các chữ số không-bull của `guess` đặt vào các vị trí không-bull và lấy số khớp lớn nhất. Dùng được cho `n <= 7`.
#### Code demo

In [15]:
def oracle(secret, guess):
    n = len(secret)
    bulls = sum(1 for i in range(n) if secret[i] == guess[i])
    free = [i for i in range(n) if secret[i] != guess[i]]
    guess_digits = [guess[i] for i in free]
    best = 0
    for perm in set(permutations(guess_digits)):                 # mọi cách sắp xếp lại các chữ số không-bull của lời đoán
        matches = sum(1 for pos, d in zip(free, perm) if secret[pos] == d)
        best = max(best, matches)
    return f"{bulls}A{best}B"


impls = {
    "list": get_hint_list,
    "counter": get_hint_counter,
    "one_pass": get_hint_one_pass,
    "two_arrays": get_hint_two_arrays,
    "sorted": get_hint_sorted,
    "submit": sol.getHint,
}

for secret, guess, expected in examples:
    assert oracle(secret, guess) == expected
    for key, fn in impls.items():
        assert fn(secret, guess) == expected, (key, secret, guess, fn(secret, guess), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [16]:
edge_cases = [
    ("1", "1", "1A0B", "một chữ số, bull"),
    ("1", "2", "0A0B", "một chữ số, không khớp"),
    ("12", "21", "0A2B", "hoán vị hoàn toàn: toàn cow"),
    ("1122", "2211", "0A4B", "trùng lặp, hoán vị hoàn toàn"),
    ("1111", "1111", "4A0B", "toàn bull"),
    ("1111", "1110", "3A0B", "ba bull, một chữ số không có trong bí mật"),
    ("1123", "0111", "1A1B", "ví dụ 2: chữ số trùng, chỉ một cow"),
    ("0000", "0000", "4A0B", "số 0 đứng đầu vẫn hợp lệ (chuỗi)"),
    ("1807", "7810", "1A3B", "ví dụ 1"),
    ("123", "456", "0A0B", "không chung chữ số nào"),
    ("11", "11", "2A0B", "hai chữ số trùng, cả hai bull"),
    ("1122", "1222", "3A0B", "ba bull, chữ số 2 dư ở lời đoán không có bản để ghép"),
]
for secret, guess, expected, label in edge_cases:
    assert oracle(secret, guess) == expected, (label, oracle(secret, guess), expected)
    for key, fn in impls.items():
        assert fn(secret, guess) == expected, (key, label, fn(secret, guess), expected)
    print("PASS", label)

PASS một chữ số, bull
PASS một chữ số, không khớp
PASS hoán vị hoàn toàn: toàn cow
PASS trùng lặp, hoán vị hoàn toàn
PASS toàn bull
PASS ba bull, một chữ số không có trong bí mật
PASS ví dụ 2: chữ số trùng, chỉ một cow
PASS số 0 đứng đầu vẫn hợp lệ (chuỗi)
PASS ví dụ 1
PASS không chung chữ số nào
PASS hai chữ số trùng, cả hai bull
PASS ba bull, chữ số 2 dư ở lời đoán không có bản để ghép


### 12.2. Tính chất và phản ví dụ

- `bulls + cows <= n`.
- **Đối xứng:** hoán đổi `secret` và `guess` cho cùng `bulls` và cùng `cows` (cả hai đối xứng).
- **Hoán vị cùng lúc** mọi vị trí của cả hai chuỗi theo cùng một phép hoán vị không đổi kết quả.
- Nếu `guess == secret` thì `nA0B`.
- **Phản ví dụ** cho cách tính cow bằng tập hợp (`in`): `"1123"`/`"0111"` cho `1A2B` thay vì `1A1B`.
#### Code demo

In [17]:
rng = random.Random(299)
for _ in range(500):
    n = rng.randint(1, 30)
    secret = "".join(rng.choice("0123456789") for _ in range(n))
    guess = "".join(rng.choice("0123456789") for _ in range(n))
    hint = sol.getHint(secret, guess)
    b, c = int(hint.split("A")[0]), int(hint.split("A")[1][:-1])
    assert b + c <= n
    assert sol.getHint(guess, secret) == hint                                  # đối xứng
    perm = list(range(n))
    rng.shuffle(perm)
    assert sol.getHint("".join(secret[i] for i in perm), "".join(guess[i] for i in perm)) == hint
    assert sol.getHint(secret, secret) == f"{n}A0B"
print("PASS tính chất (cận, đối xứng, hoán vị đồng thời, tự khớp) trên 500 cặp ngẫu nhiên")


def sai_dung_tap_hop(secret, guess):
    bulls = sum(1 for a, b in zip(secret, guess) if a == b)
    cows = sum(1 for a, b in zip(secret, guess) if a != b and b in secret)     # SAI: bỏ qua số lần xuất hiện
    return f"{bulls}A{cows}B"


print("phản ví dụ '1123','0111': đúng", sol.getHint("1123", "0111"), "| dùng 'in':", sai_dung_tap_hop("1123", "0111"))
assert sai_dung_tap_hop("1123", "0111") != sol.getHint("1123", "0111")

PASS tính chất (cận, đối xứng, hoán vị đồng thời, tự khớp) trên 500 cặp ngẫu nhiên
phản ví dụ '1123','0111': đúng 1A1B | dùng 'in': 1A2B


### 12.3. Đối chiếu ngẫu nhiên (có seed) và `n` lớn
#### Code demo

In [18]:
rng = random.Random(4299)
for trial in range(3000):
    n = rng.randint(1, 7)
    alphabet = rng.choice(["01", "0123", "0123456789"])
    secret = "".join(rng.choice(alphabet) for _ in range(n))
    guess = "".join(rng.choice(alphabet) for _ in range(n))
    truth = oracle(secret, guess)
    for key, fn in impls.items():
        assert fn(secret, guess) == truth, (key, secret, guess, fn(secret, guess), truth)
print("PASS 3000 cặp ngẫu nhiên nhỏ (bảng chữ số 2, 4 hoặc 10): mọi cách khớp oracle hoán vị")

# n = 1000 (cỡ tối đa) và n lớn hơn: so các cách với nhau
for n in (1000, 100_000):
    secret = "".join(rng.choice("0123456789") for _ in range(n))
    guess = "".join(rng.choice("0123456789") for _ in range(n))
    ref = get_hint_counter(secret, guess)
    row = []
    for key in ("counter", "one_pass", "two_arrays", "sorted", "submit") + (("list",) if n <= 1000 else ()):
        t0 = time.perf_counter()
        value = impls[key](secret, guess)
        dt = (time.perf_counter() - t0) * 1000
        assert value == ref, (key, n)
        row.append(f"{key}={dt:7.2f}ms")
    print(f"n={n:>6}: {ref} | " + " ".join(row))
print("PASS n lớn")

PASS 3000 cặp ngẫu nhiên nhỏ (bảng chữ số 2, 4 hoặc 10): mọi cách khớp oracle hoán vị
n=  1000: 84A860B | counter=   0.11ms one_pass=   0.19ms two_arrays=   0.12ms sorted=   0.32ms submit=   0.19ms list=   0.78ms
n=100000: 9985A89332B | counter=   8.97ms one_pass=  17.23ms two_arrays=  14.18ms sorted=  30.54ms submit=  17.95ms
PASS n lớn


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Tính cow bằng `in` / tập hợp (bỏ qua số lần xuất hiện)
Xem `sai_dung_tap_hop` ở phần test: `"1123"`/`"0111"` cho `1A2B` thay vì `1A1B`. **Vì sao sai:** mỗi chữ số của `secret` chỉ ghép được **một lần**; `b in secret` luôn đúng với mọi bản `1` của `guess`. **Code đúng:** đếm số lần (Counter / mảng đếm).

### Lỗi 2: Đếm bulls vào cả cows
#### Code sai

In [19]:
def sai_dem_ca_bull(secret, guess):
    bulls = sum(a == b for a, b in zip(secret, guess))
    common = sum((Counter(secret) & Counter(guess)).values())
    return f"{bulls}A{common}B"                                    # SAI: quên trừ bulls


for secret, guess, expected in examples:
    print(secret, guess, "đúng:", expected, "| sai:", sai_dem_ca_bull(secret, guess))

1807 7810 đúng: 1A3B | sai: 1A4B
1123 0111 đúng: 1A1B | sai: 1A2B


**Vì sao sai:** `common` đã gồm cả các chữ số ghép bằng đúng vị trí (bulls). Cows = `common − bulls`.

### Lỗi 3: Dùng `Counter` cộng (`+`) thay vì giao (`&`)
#### Code sai

In [20]:
def sai_cong_counter(secret, guess):
    bulls = sum(a == b for a, b in zip(secret, guess))
    merged = Counter(secret) + Counter(guess)                      # SAI: cộng số lần chứ không lấy min
    common = sum(v for v in merged.values()) // 2
    return f"{bulls}A{common - bulls}B"


for secret, guess, expected in examples:
    print(secret, guess, "đúng:", expected, "| sai:", sai_cong_counter(secret, guess))

1807 7810 đúng: 1A3B | sai: 1A3B
1123 0111 đúng: 1A1B | sai: 1A3B


**Vì sao sai:** cộng hai đa tập cho tổng số lần; ghép một–một cần `min` từng chữ số, tức **giao** (`&`).

### Lỗi 4: Sai định dạng kết quả
#### Code sai

In [21]:
def sai_dinh_dang(secret, guess):
    ans = get_hint_counter(secret, guess)
    return ans.replace("A", "a").replace("B", "b")                 # SAI: đề yêu cầu chữ hoa 'xAyB'


print(sol.getHint("1807", "7810"), sai_dinh_dang("1807", "7810"))

1A3B 1a3b


**Vì sao sai:** đầu ra phải đúng `"xAyB"` (chữ `A`, `B` hoa; số bulls trước, cows sau).

### Lỗi 5: Ghép cow trước khi loại bulls (cùng một chữ số bị dùng hai lần)
#### Code sai

In [22]:
def sai_ghep_truoc_loai_bull(secret, guess):
    bulls = sum(a == b for a, b in zip(secret, guess))
    rest = list(secret)
    cows = 0
    for g in guess:
        if g in rest:
            rest.remove(g)                                         # SAI: ghép cả chữ số đã là bull, rồi trừ bulls một lần nữa
            cows += 1
    return f"{bulls}A{cows}B"


for secret, guess, expected in examples:
    print(secret, guess, "đúng:", expected, "| sai:", sai_ghep_truoc_loai_bull(secret, guess))

1807 7810 đúng: 1A3B | sai: 1A4B
1123 0111 đúng: 1A1B | sai: 1A2B


**Vì sao sai:** nếu ghép **toàn bộ** rồi không trừ bulls (hoặc trừ sai), các chữ số bull bị đếm vào cows. Quy trình đúng: loại bulls trước (hoặc ghép toàn bộ rồi **trừ** bulls).

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**khớp một–một có trùng lặp**" (anagram, ghép phần tử) → **đa tập** (`Counter`) và **`min` số lần** (giao đa tập).
- **Bulls trước, cows sau**: loại các cặp đúng vị trí rồi mới ghép phần còn lại; hoặc `Σ min(toàn bộ) − bulls`.
- Bảng chữ cái nhỏ (`10` chữ số) → **mảng đếm** `O(1)` bộ nhớ.
- Trick một lượt: mảng đếm **cân bằng ±** cho phép ghép tức thì.
- Họ hàng: **Valid Anagram** (242), **Find All Anagrams** (438), **Intersection of Two Arrays II** (350), **Ransom Note** (383), **Minimum Number of Steps to Make Two Strings Anagram** (1347).
- Test: chữ số trùng (`1123/0111`), hoán vị hoàn toàn (`12/21`), toàn bull, không giao.

## 15. Biến thể của bài

1. **Intersection of Two Arrays II** (LeetCode 350): giao đa tập của hai mảng.
2. **Ransom Note** (383): có đủ chữ để ghép không.
3. **Mastermind** (Master Mind, CTCI 16.15): bài gần giống với số màu `4`.
4. **Đếm số lời đoán khả dĩ** nhất quán với một gợi ý (tìm kiếm trên không gian `10^n`).
5. **Gợi ý cho chữ cái** (Wordle): tô xanh/vàng/xám — cùng tư duy bull/cow.
#### Code demo

In [23]:
def wordle_feedback(secret, guess):
    """Biến thể Wordle: G (xanh, đúng vị trí), Y (vàng, có nhưng sai vị trí, tính theo số lần), '.' (xám)."""
    n = len(secret)
    result = ["."] * n
    remaining = Counter(secret[i] for i in range(n) if secret[i] != guess[i])
    for i in range(n):
        if secret[i] == guess[i]:
            result[i] = "G"
    for i in range(n):
        if result[i] == "." and remaining[guess[i]] > 0:
            result[i] = "Y"
            remaining[guess[i]] -= 1
    return "".join(result)


print(wordle_feedback("crane", "caper"), wordle_feedback("abbey", "babes"))     # in ra phản hồi cho hai cặp thử

GY.YY YYGG.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Danh sách + `remove` | loại bulls rồi ghép từng chữ số | `O(n²)` | `O(n)` | baseline |
| 2. `Counter` + giao | `Σ min(cnt) − bulls` | `O(n)` | `O(1)` | **ngắn gọn, bản nộp chuẩn** |
| 3. Một lượt, mảng cân bằng | ghép tức thì bằng `cnt` ± | `O(n)` | `O(1)` | tối ưu hằng số |
| 4. Hai mảng đếm | `min` từng chữ số | `O(n)` | `O(1)` | dễ hiểu |
| 5. Sắp xếp + hai con trỏ | giao hai dãy đã sắp | `O(n log n)` | `O(n)` | không dùng bảng đếm |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `sum((Counter("112") & Counter("121")).values())`; (b) `Solution().getHint("1234", "1234")`; (c) `Solution().getHint("12", "21")`; (d) `Solution().getHint("11", "10")`.

**Bài 2 (tính bằng tay).** Tính `getHint("1122", "1222")` và `getHint("1807", "7810")` theo hai bước: bulls rồi cows.

**Bài 3 (điền TODO).** Hoàn thiện `hint_todo(secret, guess)` bằng hai mảng đếm.
#### Code demo

In [24]:
def hint_todo(secret, guess):
    bulls = 0
    cs, cg = [0] * 10, [0] * 10
    # TODO 1: duyệt (s, g) trong zip(secret, guess)
    # TODO 2: nếu s == g: bulls += 1
    # TODO 3: cs[int(s)] += 1; cg[int(g)] += 1 (đếm mọi chữ số)
    # TODO 4: cows = sum(min(a, b) for a, b in zip(cs, cg)) - bulls
    # TODO 5: trả f"{bulls}A{cows}B"
    pass


# Tự làm xong rồi mới mở test:
# for secret, guess, expected in examples:
#     assert hint_todo(secret, guess) == expected

**Bài 4 (tự viết lại).** Viết `hint_one_pass(secret, guess)` bằng mảng cân bằng (Cách 3), **không nhìn lại** code mẫu; đối chiếu với `oracle` trên chuỗi ngắn.

**Bài 5 (biến thể / mini challenge).** Viết `consistent_guesses(n, secret_hint_pairs)` đếm số chuỗi `n` chữ số (`n <= 4`) **nhất quán** với tất cả cặp `(guess, hint)` đã cho (nghĩa là với mỗi cặp, nếu chuỗi đó là bí mật thì gợi ý phải đúng bằng `hint`). Thử `n = 2`, `[("12", "1A0B")]`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: giao đa tập lấy `min`. Bài 2: bulls trước (so vị trí), rồi cows từ phần còn lại. Bài 5: duyệt mọi chuỗi `10^n` và so sánh với `getHint`.
- **Mức 2 (vừa):** Bài 1: `3`, `"4A0B"`, `"0A2B"`, `"1A0B"`. Bài 2: `"1122"/"1222"`: bulls ở các vị trí `0`, `2`, `3` ⇒ `3`; phần còn lại (secret `'1'`, guess `'2'`) không ghép ⇒ `0` cow ⇒ `"3A0B"`. `"1807"/"7810"` là `1A3B`. Bài 3: khung Cách 4.
- **Mức 3 (sâu):** Bài 4: dùng `cnt` cân bằng (xem trace). Bài 5: duyệt mọi bí mật `n` chữ số bằng `itertools.product` và giữ những bí mật mà `getHint(bí_mật, đoán) == hint` với mọi cặp.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [25]:
# Bài 1
assert sum((Counter("112") & Counter("121")).values()) == 3
assert Solution().getHint("1234", "1234") == "4A0B"
assert Solution().getHint("12", "21") == "0A2B"
assert Solution().getHint("11", "10") == "1A0B"

# Bài 2
assert Solution().getHint("1122", "1222") == oracle("1122", "1222") == "3A0B"
assert Solution().getHint("1807", "7810") == "1A3B"


# Bài 3
def hint_todo_ans(secret, guess):
    bulls = 0
    cs, cg = [0] * 10, [0] * 10
    for s, g in zip(secret, guess):
        if s == g:
            bulls += 1
        cs[int(s)] += 1
        cg[int(g)] += 1
    cows = sum(min(a, b) for a, b in zip(cs, cg)) - bulls
    return f"{bulls}A{cows}B"


for secret, guess, expected in examples:
    assert hint_todo_ans(secret, guess) == expected


# Bài 4
def hint_one_pass(secret, guess):
    bulls = cows = 0
    cnt = [0] * 10
    for s, g in zip(secret, guess):
        if s == g:
            bulls += 1
        else:
            si, gi = int(s), int(g)
            cows += (cnt[si] < 0) + (cnt[gi] > 0)
            cnt[si] += 1
            cnt[gi] -= 1
    return f"{bulls}A{cows}B"


rng = random.Random(20)
for _ in range(300):
    n = rng.randint(1, 7)
    a = "".join(rng.choice("0123") for _ in range(n))
    b = "".join(rng.choice("0123") for _ in range(n))
    assert hint_one_pass(a, b) == oracle(a, b)


# Bài 5
def consistent_guesses(n, pairs):
    candidates = ["".join(map(str, digits)) for digits in product(range(10), repeat=n)]
    return [c for c in candidates if all(sol.getHint(c, guess) == hint for guess, hint in pairs)]


res = consistent_guesses(2, [("12", "1A0B")])
assert all(sol.getHint(c, "12") == "1A0B" for c in res) and len(res) > 0
print("số bí mật 2 chữ số nhất quán với ('12', '1A0B'):", len(res))
print("PASS đáp án bài tập")

số bí mật 2 chữ số nhất quán với ('12', '1A0B'): 18
PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm các chữ số không-bull, lấy `min`)

```python
from collections import Counter

class Solution:
    def getHint(self, secret: str, guess: str) -> str:
        cnt1, cnt2 = Counter(), Counter()
        x = 0
        for a, b in zip(secret, guess):
            if a == b:
                x += 1
            else:
                cnt1[a] += 1
                cnt2[b] += 1
        y = sum(min(cnt1[c], cnt2[c]) for c in cnt1)
        return f"{x}A{y}B"
```

**Checklist**

- [ ] Bulls: so sánh **theo vị trí**; cows: **giao đa tập** của các chữ số **không-bull**, mỗi chữ số ghép **một lần**?
- [ ] Không dùng `in` / tập hợp (bỏ qua số lần xuất hiện)?
- [ ] Không đếm bulls vào cows (hoặc trừ `bulls` nếu đếm toàn bộ)?
- [ ] Định dạng `"xAyB"` (A, B hoa; bulls trước)?
- [ ] Đã thử `1123/0111`, hoán vị hoàn toàn, toàn bull, không giao, chữ số `0` đầu?
- [ ] Đã đối chiếu với brute force hoán vị các chữ số không-bull?
- [ ] Skeleton đúng `class Solution` / `def getHint(self, secret: str, guess: str) -> str`?